# Detection significance: comparing methods

This notebook gathers the different ways of turning a Kp-Vsys logL grid into a
detection-significance statement ("the signal is detected at N sigma"), and
explains the reasoning behind each one -- including the pitfalls. It complements
`kpvsys_explained.ipynb` (which explains the logL/CCF math itself) and
`kpvsys_products.ipynb` (which shows the standard marginalized-posterior workflow).

**Prerequisite:** run `kpvsys_products.ipynb` first (or at least its loading cells)
to understand `alpha_frac`, `idx_signal`, and the marginalized Kp-Vsys posterior --
this notebook assumes that context and does not re-explain it.

**Contents**
1. The problem: four different notions of "detected"
2. Method 1 -- geometric: sigma contours on the marginalized posterior (already in use)
3. Method 2 -- Bayesian: marginalized Bayes factor (and the prior pitfall)
4. Method 3 -- frequentist: profile likelihood-ratio test
5. Method 4 -- empirical: box / sigma-clipping noise floor
6. Side-by-side comparison
7. Practical recommendation

In [ ]:
from pathlib import Path

import numpy as np
from scipy import stats

import starships.logl_grid as lg
from starships.plotting_fcts import plot_kpvsys_map, find_isolated_peak_sigma, plot_2d_map

## 0. Load a grid

Same loading pattern as `kpvsys_products.ipynb`. `idx_signal` selects the exposures
where the planet signal is expected (`alpha_frac > 0.5`).

In [ ]:
path_results = Path.home() / 'scratch/DataAnalysis/SPIRou/logl_grids/'
file_glob = 'take3_HRR_modif_disso_31254924_all_kp*_visit*.npz'

alpha_array = np.linspace(0.01, 2., 31)

files = sorted(path_results.glob(file_glob))
lg.load_logl_results(files)

alpha_frac = lg._loaded_extra['alpha_frac']
(idx_signal,) = np.nonzero(alpha_frac > 0.5)
print(f'Exposures with planetary signal: {len(idx_signal)} / {len(alpha_frac)}')

## 1. The problem: four different notions of "detected"

The logL grid gives us $\log\mathcal{L}(v_\mathrm{sys}, K_p, \alpha)$ -- a number at
every point of a 3D grid. The amplitude $\alpha$ scales the model spectrum (see
`kpvsys_explained.ipynb` §5 for why $\alpha$ exists and why we don't know it ahead of
time: $\alpha=1$ means the model is perfectly calibrated, $\alpha<1$ means the real
signal is weaker than predicted).

"Is there a detection?" always boils down to comparing two situations:
- **with a planet signal** ($\alpha > 0$, at some $v_\mathrm{sys}$, $K_p$)
- **without one** ($\alpha = 0$, the null hypothesis -- pure noise, no planet term
  at all)

but there are genuinely different, equally legitimate ways to make that comparison
precise (or, for Method 4, to sidestep it entirely), and they can give different
numbers on the same data. None of them is "the" right answer -- they answer subtly
different questions. This notebook walks through four of them so you can pick (or
report several) with your eyes open.

## 2. Method 1 -- geometric: sigma contours on the marginalized posterior

**This is the method already used in `kpvsys_products.ipynb`.** It starts from the
alpha-marginalized posterior $P(K_p, v_\mathrm{sys}\,|\,\mathrm{data})$ (already
normalized to sum to 1 over the grid) and asks: *how much of the total probability
mass is packed into a small region around the peak?* Concretely,
`find_isolated_peak_sigma` sorts all grid cells by probability, accumulates mass from
the peak outward, and finds how many "sigma" (in the usual Gaussian sense: 1 sigma =
68.3% mass, 2 sigma = 95.4%, ...) it takes to enclose the peak before hitting a
region as likely as the noise floor.

**What it captures well:** whether the peak is *isolated* -- a sharp, lonely peak
gets a high sigma; a broad or multi-modal posterior gets a low one, even if the peak
value itself is high. This is intuitive and matches how the map *looks* by eye.

**What it doesn't give you:** a formal hypothesis test in the statistical sense (no
p-value against a null distribution, no reference to $\alpha=0$ at all -- it only
looks at the *shape* of the posterior, not at how it compares to a no-signal
baseline). Two different datasets could have the same sigma from this method while
having very different amounts of *actual* evidence for a planet, if their noise
properties differ.

In [ ]:
posterior, vsys_os, kp_os, margin_vsys, margin_kp = lg.compute_kpvsys_posterior(
    alpha_array=alpha_array, idx_signal=idx_signal, oversample=2,
)

sigma_geometric = find_isolated_peak_sigma(posterior, vsys_os, kp_os)
print(f'Method 1 (geometric): {sigma_geometric:.2f} sigma')

fig, axes = plot_kpvsys_map(
    posterior, vsys_os, kp_os, margin_vsys, margin_kp,
    n_sigma=3, sigma_display='contours',
)

## 3. Method 2 -- Bayesian: marginalized Bayes factor

A **Bayes factor** directly compares the two models via the ratio of their
*evidences* -- how well each model, averaged over everything it doesn't know a
priori, predicts the data:

$$\mathrm{BF} = \frac{p(\mathrm{data}\,|\,\mathrm{signal\ model})}
{p(\mathrm{data}\,|\,\mathrm{null\ model})}$$

The signal model has three free parameters it doesn't know ahead of time --
$\alpha$, $v_\mathrm{sys}$, $K_p$ -- so its evidence is obtained by *averaging* the
likelihood over a prior for each of them (here, uniform: "any value in this range is
equally likely a priori"):

$$p(\mathrm{data}\,|\,\mathrm{signal}) = \int\!\!\int\!\!\int
\mathcal{L}(v_\mathrm{sys}, K_p, \alpha)\,
p(\alpha)\,p(v_\mathrm{sys})\,p(K_p)\; d\alpha\,dv_\mathrm{sys}\,dK_p$$

The null model has *no* free parameters (it's just $\alpha=0$, a single fixed point),
so its "evidence" is simply its logL. `log10(BF)` is usually read off the Jeffreys
scale: below 0.5, not worth mentioning; 0.5-1, substantial; 1-2, strong; above 2,
decisive evidence for the signal model.

In [ ]:
bf_result = lg.compute_kpvsys_bayes_factor(alpha_array=alpha_array, idx_signal=idx_signal)
print(f"Method 2 (Bayes factor): log10(BF) = {bf_result['log_bf'] / np.log(10):.2f}")
print(f"  alpha prior range: [{alpha_array[0]:.2f}, {alpha_array[-1]:.2f}] -> width {bf_result['alpha_range']:.2f}")

### 3a. The prior pitfall -- why the range you pick matters

Averaging the likelihood over a prior, instead of just evaluating it at the best
point, introduces a term of the form $1 / (\text{range of the prior})$ into the
evidence -- an **Occam factor**. It's what makes a Bayes factor penalize a model
that hedges its bets over a huge range of possible amplitudes/velocities: if most of
that range doesn't fit the data at all, the model gets "punished" for having wasted
prior mass there, even though *some* value in that range fits perfectly.

This is not a bug -- it's the entire point of a Bayes factor, and it's a large part
of why Bayes factors are considered more conservative than a naive "best fit"
comparison. **But it also means the number you get genuinely depends on a choice you
made**, not only on the data. `compute_kpvsys_bayes_factor` applies this Occam
correction for $\alpha$, $v_\mathrm{sys}$, *and* $K_p$ (all three are free
parameters of the signal model that the null model simply doesn't have).

**Two different things can make log(BF) go down when you narrow a range -- don't
confuse them:**
- **Truncation.** The narrower range excludes where the likelihood is actually high
  (e.g. an `alpha_array` that stops before the true best-fit alpha). log(BF) drops
  because you literally left out real evidence -- this says nothing about
  robustness, it's just the wrong range for this data. *Always check the range
  covers `compute_alpha_significance`'s `alpha_best` (and `vsys_best`/`kp_best` for
  the bounds in §3b below) before reading anything into a lower log(BF).*
- **Genuine Occam dilution.** The range already fully covers the peak, and is
  simply made wider still by adding "empty" territory the data disfavor. *This* is
  the sensitivity effect actually worth reporting.

The cell below makes the second effect concrete: same data, same grid, only the
assumed $\alpha$ range changes -- watch for the first effect (truncation) when you
read the result.

In [ ]:
print('Sensitivity of log10(BF) to the alpha prior range:')
for a_max in [1., 2., 5., 10.]:
    bf_i = lg.compute_kpvsys_bayes_factor(
        alpha_array=np.linspace(0.01, a_max, 31), idx_signal=idx_signal,
    )
    print(f"  alpha in [0.01, {a_max:>4.1f}]:  log10(BF) = {bf_i['log_bf'] / np.log(10):6.2f}")

If a range **fully covers** where the likelihood has support, widening it further
should move log10(BF) only modestly for a strong, well-localized signal (roughly
`log10(range_2 / range_1)`), and a lot more for a marginal detection -- which is
itself useful information: it tells you the "decisive" reading you got with one
particular range is not robust.

**But if a narrower range doesn't cover the peak at all, a lower log(BF) there is
truncation, not a sensitivity result.** This actually happened on a real KELT-20b
grid: `alpha_best` came out at 3.24, so `alpha_array=[0.01, 2]` cut off the peak
entirely and gave a *lower* log10(BF) (1.50) than `[0.01, 7]` (2.35) -- not because
$[0.01,2]$ is a "more conservative" prior, but simply because it excluded the
region the data actually supports. Widening further to `[0.01, 15]` then showed the
genuine dilution effect: log10(BF) came back down slightly (2.02), because $[7, 15]$
is mostly empty territory. **Moral: check `alpha_best` is inside your range before
comparing anything.**

### 3b. Restricting vsys/Kp too

The same Occam mechanism applies to $v_\mathrm{sys}$ and $K_p$: by default
`compute_kpvsys_bayes_factor` treats the *entire loaded grid* as their prior
range. `vsys_bounds`/`kp_bounds` let you restrict the integral (and its prior
normalization) to a smaller window -- technically just as simple as narrowing
`alpha_array`.

**This is much riskier than restricting alpha, though.** Narrowing vsys/Kp is
only a legitimate prior if the bounds come from information *independent of this
grid* -- e.g. an orbital-dynamics constraint on $K_p$ known ahead of time. If you
instead look at where the peak landed on the full map and *then* draw a box
around it, you are not testing a real alternative model any more -- you're
reporting the evidence for "a signal is exactly where I already saw it", which is
circular (the classic look-elsewhere-effect failure mode, just applied to a
Bayes factor instead of a p-value). If you can't justify the bounds without
pointing at this map, don't use them here.

Also watch for the same truncation trap as in §3a: bounds that don't fully cover
`vsys_best`/`kp_best` (from `compute_alpha_significance`) will lower log(BF) by
truncation, not by a meaningful Occam effect.

In [ ]:
# Where does the peak actually sit? Needed to check the bounds below cover it.
# (a fresh call here -- this section is self-contained and doesn't depend on
# Method 3 below, even though both use compute_alpha_significance)
peak_check = lg.compute_alpha_significance(idx_signal=idx_signal)
print(f"Peak location: vsys_best={peak_check['vsys_best']:.1f}, "
      f"Kp_best={peak_check['kp_best']:.1f}")

# Example ONLY valid if these bounds come from something other than this map
# (e.g. a known orbital constraint on Kp) -- see the warning above.
bf_restricted = lg.compute_kpvsys_bayes_factor(
    alpha_array=alpha_array, idx_signal=idx_signal,
    vsys_bounds=(-30., 30.), kp_bounds=(100., 250.),
)
print(f"Full grid:          log10(BF) = {bf_result['log_bf'] / np.log(10):.2f}  "
      f"(vsys_range={bf_result['vsys_range']:.0f}, kp_range={bf_result['kp_range']:.0f})")
print(f"Restricted vsys/Kp: log10(BF) = {bf_restricted['log_bf'] / np.log(10):.2f}  "
      f"(vsys_range={bf_restricted['vsys_range']:.0f}, "
      f"kp_range={bf_restricted['kp_range']:.0f})")

## 4. Method 3 -- frequentist: profile likelihood-ratio test

This method sidesteps the prior question entirely by not averaging over $\alpha$ at
all. Instead, it finds the single best-fitting point $(\alpha_\mathrm{best},
v_{\mathrm{sys,best}}, K_{p,\mathrm{best}})$ -- the maximum-likelihood estimate, or
MLE -- and directly compares its logL to the null's:

$$D = 2\left(\log\mathcal{L}_\mathrm{best} - \log\mathcal{L}_{\alpha=0}\right)$$

Intuitively: *how much better does the single best model with a planet explain the*
*data, compared to no planet at all?* No prior range to choose -- but the trade-off
is that we've switched from a Bayesian question ("how much should this data update my
belief") to a frequentist one ("how surprising would this data be if there were truly
no signal").

### Why the usual chi-square recipe needs a correction here

For this kind of comparison, a classic result (Wilks' theorem) says that $D$ should
follow a chi-square distribution with 1 degree of freedom *if the null hypothesis is
true* -- which lets you convert $D$ directly into a p-value, then into a sigma.

**But** that classic result assumes $\alpha$ is free to be *any* real number,
including negative. Here $\alpha \ge 0$ always (you can't have a negative amount of
planet signal) -- so $\alpha=0$ sits right at the *edge* of what's allowed, not in
the middle. When the tested parameter is pinned to a boundary like this, the correct
reference distribution (Chernoff's theorem) is not a plain chi-square: it's a 50/50
mix of "always exactly 0" and "a chi-square with 1 degree of freedom". In practice
this means: **the p-value is half of what the naive chi-square recipe would give you**
(`0.5 * chi2(df=1).sf(D)` instead of `1.0 * chi2(df=1).sf(D)`). Skipping this halving
silently overstates how significant the detection looks.

In [ ]:
sig_result = lg.compute_alpha_significance(idx_signal=idx_signal)
print('Method 3 (likelihood-ratio):')
print(f"  alpha_best = {sig_result['alpha_best']:.2f},  "
      f"vsys_best = {sig_result['vsys_best']:.1f} km/s,  "
      f"Kp_best = {sig_result['kp_best']:.1f} km/s")
print(f"  D = {sig_result['D']:.2f}   p-value = {sig_result['p_value']:.2e}   -> {sig_result['sigma']:.2f} sigma")

### 4a. Visualizing the fixed-alpha map

`compute_kpvsys_posterior_fixed_alpha` gives the same kind of Kp-Vsys map as
`compute_kpvsys_posterior`, but at a single, fixed $\alpha$ instead of marginalized
over it -- for instance, at the MLE found above. This is a *display* tool, not a
detection statistic on its own; it's useful to see where on the grid
$\alpha_\mathrm{best}$ actually lives.

In [ ]:
posterior_fixed, vsys_f, kp_f, mv_f, mk_f = lg.compute_kpvsys_posterior_fixed_alpha(
    alpha=sig_result['alpha_best'], idx_signal=idx_signal,
)
fig, axes = plot_kpvsys_map(
    posterior_fixed, vsys_f, kp_f, mv_f, mk_f, n_sigma=3, sigma_display='contours',
)
axes[0].set_title(f"alpha fixed at MLE = {sig_result['alpha_best']:.2f}")

### 4b. A sigma map instead of a single peak

`compute_alpha_significance` only reports the statistic at its own best
point. `compute_alpha_significance_map` computes the *exact same*
Chernoff-corrected likelihood-ratio sigma at **every** (vsys, Kp) grid
point instead -- same formula, same closed-form alpha MLE, just evaluated
everywhere rather than once. The two agree exactly at the map's own peak.

**The look-elsewhere caveat is not weaker here.** A whole region reading
above, say, 3 sigma is not stronger evidence than the single best point
already gives on its own -- it's the same one detection's neighbourhood,
not several independent ones. Don't add them up.

In [ ]:
sig_map_result = lg.compute_alpha_significance_map(idx_signal=idx_signal)
print(f"Peak: {sig_map_result['peak_sigma']:.2f} sigma at "
      f"vsys={sig_map_result['vsys_best']:.1f}, Kp={sig_map_result['kp_best']:.1f} "
      f"(alpha_best={sig_map_result['alpha_best']:.2f})")


def plot_sigma_map_with_slices(result, **kwargs):
    """Small local helper, reused for Method 4 below -- not library API, just
    a few lines wired together, copy/adapt freely. CCF/sigma maps aren't
    marginalizable like a true posterior, so the side panels show a SLICE
    through the map at the peak instead of a marginal; `plot_2d_map` doesn't
    care what the 1D curves mean, it just plots what it's given."""
    sigma_map = result['sigma_map']
    vc, kc = result['vsys_coords'], result['kp_coords']
    i_v = int(np.argmin(np.abs(vc - result['vsys_best'])))
    i_k = int(np.argmin(np.abs(kc - result['kp_best'])))
    return plot_2d_map(
        sigma_map, vc, kc,
        margin_x=sigma_map[:, i_k], margin_y=sigma_map[i_v, :],
        mark_peak=True, scale='linear', cbar_label=r'$\sigma$', **kwargs,
    )


fig, (ax_map, ax_y, ax_x) = plot_sigma_map_with_slices(sig_map_result)
fig.suptitle('Likelihood-ratio sigma map', y=1.02)

## 5. Method 4 -- empirical: box / sigma-clipping noise floor

A method widely used in the HRCCS literature (Brogi/Snellen-style CCF maps):
instead of any formal probability comparison, just measure how many standard
deviations the map's peak sits above a noise floor estimated **from the map
itself**. Two ways to estimate that noise floor:
- **`method='box'`**: standard deviation inside a region you believe is
  signal-free.
- **`method='clip'`**: iteratively sigma-clip the *whole* map (removing
  outliers -- including the peak -- each round) and use the standard
  deviation of what survives.

Both give `sigma_map = (map - median) / noise_std`; the peak's value on this
map is the reported "N sigma". **This has no formal null-hypothesis test
behind it** -- unlike Methods 2/3, there's no reference to $\alpha=0$ and no
asymptotic distribution result. It works well in practice as long as the
estimated noise floor is actually representative and roughly Gaussian, but
that's an assumption, not something the method verifies for you.

Ported here (generalized to work on any `(vsys, Kp)` map instead of the older
`correlation_class.Correlations` CCF object) from code originally written by
Joost Wardenier and improved by Mathis Bouffard, June 2025.

In [ ]:
ccf_map = lg.get_ccf(idx_exposure=idx_signal, sum_axis=(-2, -1))

empirical_result = lg.compute_empirical_sigma_map(ccf_map, method='clip')
print(f"Method 4 (clip): {empirical_result['peak_sigma']:.2f} sigma  "
      f"(vsys={empirical_result['vsys_best']:.1f}, "
      f"Kp={empirical_result['kp_best']:.1f})")

In [ ]:
fig, (ax_map, ax_y, ax_x) = plot_sigma_map_with_slices(empirical_result)
fig.suptitle('Empirical sigma map (sigma-clip, CCF)', y=1.02)

**A pitfall worth hitting once: don't feed this the *linear* posterior.**
`map_2d` must be a CCF/logL-*scale* map with roughly homogeneous background
variance -- `get_ccf(...)` above qualifies. `compute_kpvsys_posterior`'s output
does not, *as long as it stays linear*: it is `exp(logL - max)`, normalized to
max=1, so almost the entire map away from the peak is squashed to ~0 by
construction (numerical underflow, not a real noise floor). Box/clip would then
measure that underflow instead of actual noise, and the peak comes out at an
absurd, meaningless number of "sigma" -- this is **not** a resolution/oversampling
issue (plain `oversample=1` has the exact same problem); it's the exponentiation
itself that breaks the method's assumption. The cell below reproduces this on
purpose, so the failure mode is recognizable if you ever see it by accident.

In [ ]:
# DON'T DO THIS -- reproduced on purpose, see the warning above.
broken_result = lg.compute_empirical_sigma_map(
    posterior, method='clip', vsys_coords=vsys_os, kp_coords=kp_os,
)
print(f"Method 4 misapplied to the posterior: "
      f"{broken_result['peak_sigma']:.0f} sigma  <-  meaningless, ignore this")
print('The correct input for Method 4 is a CCF/logL-scale map, e.g. ccf_map above.')

### 5b. The fix -- use the *log*-posterior instead

Taking `np.log(posterior)` undoes the exponentiation and recovers a
logL-like quantity -- which is, not coincidentally, exactly what this
module's plots already show by default (`plot_posterior_2d`'s
`scale='log'`). This works as long as `posterior` hasn't underflowed to
exact `0.0` anywhere (check `np.isfinite(np.log(posterior)).all()` first --
non-finite pixels are automatically excluded from the noise-floor estimate,
but if most of the map is non-finite there isn't enough left to measure a
noise floor from).

Note this gives a genuinely *different* number from Method 4 on the raw CCF
map above -- `posterior` has already been marginalized over alpha, a more
statistically-processed quantity than a raw CCF sum, so there's no reason
to expect the two to match. Neither is "more correct" than the other; they're
cross-checks on different intermediate products.

In [ ]:
log_posterior = np.log(posterior)
print(f'All finite: {np.isfinite(log_posterior).all()}')

fixed_result = lg.compute_empirical_sigma_map(
    log_posterior, method='clip', vsys_coords=vsys_os, kp_coords=kp_os,
)
print(f"Method 4 (clip, on log(posterior)): "
      f"{fixed_result['peak_sigma']:.2f} sigma")

In [ ]:
fig, (ax_map, ax_y, ax_x) = plot_sigma_map_with_slices(fixed_result)
fig.suptitle('Empirical sigma map (sigma-clip, log(posterior))', y=1.02)

### 5c. The box variant -- and the same double-dipping warning as §3b

`method='box'` needs a region you're confident is signal-free. **Choosing that
region by looking at this exact map and picking a spot away from the peak is
the same look-elsewhere problem flagged for `vsys_bounds`/`kp_bounds` in
§3b** -- it's only a legitimate noise estimate if the region is justified
independently (e.g. a velocity range known to be outside any physically
plausible $K_p$). The cell below picks a corner of the loaded grid purely for
illustration -- treat the resulting number accordingly, not as a citable
figure.

In [ ]:
vsys_lo, vsys_hi = vsys_os.min(), vsys_os.max()
kp_lo, kp_hi = kp_os.min(), kp_os.max()
# Illustration only -- a real box must be justified independently of this map.
box_result = lg.compute_empirical_sigma_map(
    ccf_map, method='box',
    box_vsys=(vsys_lo, vsys_lo + 0.2 * (vsys_hi - vsys_lo)),
    box_kp=(kp_lo, kp_lo + 0.2 * (kp_hi - kp_lo)),
)
print(f"Method 4 (box, illustrative): {box_result['peak_sigma']:.2f} sigma")

In [ ]:
fig, (ax_map, ax_y, ax_x) = plot_sigma_map_with_slices(box_result)
fig.suptitle('Empirical sigma map (box, illustrative)', y=1.02)

# plot_2d_map doesn't know about the noise-floor box (that's specific to
# method='box') -- just draw it directly.
bx, bk = box_result['box_vsys'], box_result['box_kp']
rect_x = [bx[0], bx[1], bx[1], bx[0], bx[0]]
rect_y = [bk[0], bk[0], bk[1], bk[1], bk[0]]
ax_map.plot(rect_x, rect_y, color='w', linestyle='--', linewidth=1.5)

## 6. Side-by-side comparison

| | Method 1: geometric | Method 2: Bayes factor | Method 3: likelihood-ratio | Method 4: empirical |
|---|---|---|---|---|
| **Framework** | descriptive (shape of the posterior) | Bayesian | frequentist | empirical / ad hoc |
| **Compares to $\alpha=0$ explicitly?** | no | yes | yes | no |
| **Needs a prior on $\alpha$?** | no (but marginalizes with one internally) | yes -- result depends on the chosen range | no | no |
| **Needs a prior on $v_\mathrm{sys}$, $K_p$?** | no | yes (the loaded grid's range) | no | no (but needs a signal-free box, or trusts sigma-clipping) |
| **Sensitive to peak shape only, or to absolute fit quality?** | shape only | absolute (properly normalized) | absolute (single best point) | absolute, but vs. an assumed-Gaussian empirical noise floor, not $\alpha=0$ |
| **Known correction needed** | none | report range sensitivity | Chernoff boundary mixture (0.5 factor) | none -- but no formal test underneath, so treat as a cross-check |
| **Output** | sigma (geometric) | log10(BF), Jeffreys scale | sigma (statistical) | sigma (empirical) |

In [ ]:
print('Summary for this dataset:')
print(f'  Method 1 (geometric):        {sigma_geometric:.2f} sigma')
print(f"  Method 2 (Bayes factor):     log10(BF) = {bf_result['log_bf'] / np.log(10):.2f}")
print(f"  Method 3 (likelihood-ratio): {sig_result['sigma']:.2f} sigma")
print(f"  Method 4 (empirical, clip):  {empirical_result['peak_sigma']:.2f} sigma")

## 7. Practical recommendation

- The four methods answer different questions and **are not expected to give
  identical numbers** -- a large disagreement is worth investigating (e.g. a very
  sharp but small peak can look great geometrically while carrying weak absolute
  evidence), but a moderate one is normal and not a bug.
- For a methods section, **report more than one number**, and say explicitly which
  one is your headline figure and why. A geometric sigma alone, without a
  comparison to $\alpha=0$, is easy for a referee to push back on.
- If you report a Bayes factor, **always state the prior range** you integrated over
  and ideally show the sensitivity check from §3a -- an unstated prior range makes
  the number impossible to reproduce or scrutinize.
- If you report a likelihood-ratio sigma, **make sure the Chernoff correction (the
  0.5 factor) is applied** -- `compute_alpha_significance` already does this, but if
  you ever recompute $D$ by hand, don't forget it.
- If you report Method 4's empirical sigma, **say which noise-floor method you
  used and why** (box region / sigma-clip settings) -- it has no formal
  null-hypothesis derivation behind it, so its credibility rests entirely on the
  noise floor actually being representative and Gaussian. Prefer it as a
  cross-check alongside Method 2/3, not as your only reported number.
- Method 2's $v_\mathrm{sys}$/$K_p$ bounds (§3b), Method 3's
  $\alpha_\mathrm{best}$/$v_\mathrm{sys,best}$/$K_{p,\mathrm{best}}$, and Method 4's
  box region all risk the same **look-elsewhere double-dipping** if chosen *after*
  looking at where the peak landed on this exact map -- justify them independently
  of this grid, or don't narrow/restrict at all.